# 高级模型服务系统教程

本教程涵盖：
1. 熔断器模式
2. 重试策略
3. 分布式追踪
4. 指标收集
5. 安全最佳实践

In [ ]:
import asyncio
import time
import random
import json
from enum import Enum
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Callable
from collections import defaultdict
import numpy as np

## 1. 熔断器模式

防止级联故障，快速失败。

In [ ]:
class CircuitState(Enum):
    CLOSED = "closed"      # 正常
    OPEN = "open"          # 熔断
    HALF_OPEN = "half_open"  # 半开

class CircuitBreaker:
    def __init__(self, failure_threshold=5, recovery_timeout=10):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.last_failure_time = 0
        self.success_count = 0
    
    def can_execute(self):
        if self.state == CircuitState.CLOSED:
            return True
        elif self.state == CircuitState.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = CircuitState.HALF_OPEN
                print(f'Circuit: OPEN -> HALF_OPEN')
                return True
            return False
        return True  # HALF_OPEN
    
    def record_success(self):
        if self.state == CircuitState.HALF_OPEN:
            self.success_count += 1
            if self.success_count >= 3:
                self.state = CircuitState.CLOSED
                self.failure_count = 0
                self.success_count = 0
                print(f'Circuit: HALF_OPEN -> CLOSED')
    
    def record_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        
        if self.state == CircuitState.HALF_OPEN:
            self.state = CircuitState.OPEN
            print(f'Circuit: HALF_OPEN -> OPEN')
        elif self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN
            print(f'Circuit: CLOSED -> OPEN (failures={self.failure_count})')

# 测试熔断器
cb = CircuitBreaker(failure_threshold=3, recovery_timeout=2)

def unreliable_service():
    if random.random() < 0.7:
        raise Exception('Service failed')
    return 'Success'

for i in range(15):
    if cb.can_execute():
        try:
            result = unreliable_service()
            cb.record_success()
            print(f'Request {i}: {result}')
        except:
            cb.record_failure()
            print(f'Request {i}: Failed')
    else:
        print(f'Request {i}: Circuit OPEN, fast fail')
    time.sleep(0.5)

## 2. 重试策略

In [ ]:
class RetryStrategy:
    @staticmethod
    def exponential_backoff(attempt, base_delay=1.0, max_delay=30.0):
        delay = min(base_delay * (2 ** attempt), max_delay)
        jitter = random.uniform(0, delay * 0.1)
        return delay + jitter
    
    @staticmethod
    def linear_backoff(attempt, base_delay=1.0, max_delay=30.0):
        return min(base_delay * (attempt + 1), max_delay)

def retry_with_backoff(func, max_retries=3, strategy='exponential'):
    for attempt in range(max_retries + 1):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries:
                raise
            if strategy == 'exponential':
                delay = RetryStrategy.exponential_backoff(attempt)
            else:
                delay = RetryStrategy.linear_backoff(attempt)
            print(f'Attempt {attempt+1} failed, retrying in {delay:.2f}s')
            time.sleep(delay)

# 测试
call_count = [0]
def flaky_service():
    call_count[0] += 1
    if call_count[0] < 3:
        raise Exception('Temporary failure')
    return 'Success after retries'

result = retry_with_backoff(flaky_service, max_retries=5)
print(f'Result: {result}')

## 3. 指标收集器

In [ ]:
class MetricsCollector:
    def __init__(self):
        self.counters = defaultdict(int)
        self.histograms = defaultdict(list)
        self.gauges = {}
    
    def inc_counter(self, name, labels=None, value=1):
        key = (name, str(labels) if labels else '')
        self.counters[key] += value
    
    def observe_histogram(self, name, value, labels=None):
        key = (name, str(labels) if labels else '')
        self.histograms[key].append(value)
    
    def set_gauge(self, name, value, labels=None):
        key = (name, str(labels) if labels else '')
        self.gauges[key] = value
    
    def get_summary(self):
        summary = {'counters': dict(self.counters), 'gauges': dict(self.gauges)}
        summary['histograms'] = {}
        for key, values in self.histograms.items():
            if values:
                summary['histograms'][str(key)] = {
                    'count': len(values),
                    'mean': np.mean(values),
                    'p50': np.percentile(values, 50),
                    'p99': np.percentile(values, 99),
                }
        return summary

# 使用示例
metrics = MetricsCollector()

for i in range(100):
    latency = random.uniform(0.01, 0.1)
    status = 'success' if random.random() > 0.1 else 'error'
    
    metrics.inc_counter('requests_total', {'status': status})
    metrics.observe_histogram('latency_seconds', latency)

metrics.set_gauge('active_connections', 42)

print('Metrics Summary:')
for k, v in metrics.get_summary().items():
    print(f'  {k}: {v}')

## 4. 速率限制器

In [ ]:
class SlidingWindowRateLimiter:
    """滑动窗口速率限制"""
    def __init__(self, requests_per_second=10):
        self.rate = requests_per_second
        self.window_size = 1.0
        self.requests = defaultdict(list)
    
    def is_allowed(self, client_id: str) -> bool:
        now = time.time()
        window_start = now - self.window_size
        
        self.requests[client_id] = [
            t for t in self.requests[client_id] if t > window_start
        ]
        
        if len(self.requests[client_id]) >= self.rate:
            return False
        
        self.requests[client_id].append(now)
        return True

class TokenBucketRateLimiter:
    """令牌桶速率限制"""
    def __init__(self, rate=10, capacity=20):
        self.rate = rate
        self.capacity = capacity
        self.tokens = capacity
        self.last_update = time.time()
    
    def is_allowed(self) -> bool:
        now = time.time()
        elapsed = now - self.last_update
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_update = now
        
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

# 测试
limiter = TokenBucketRateLimiter(rate=5, capacity=10)

allowed = 0
denied = 0
for _ in range(20):
    if limiter.is_allowed():
        allowed += 1
    else:
        denied += 1

print(f'Allowed: {allowed}, Denied: {denied}')

## 5. 输入验证与安全

In [ ]:
class InputValidator:
    """输入验证器"""
    @staticmethod
    def validate_numeric_array(data, min_len=1, max_len=10000, min_val=-1e6, max_val=1e6):
        if not isinstance(data, (list, np.ndarray)):
            raise ValueError('Data must be a list or array')
        
        arr = np.array(data, dtype=np.float32)
        
        if len(arr) < min_len or len(arr) > max_len:
            raise ValueError(f'Data length must be between {min_len} and {max_len}')
        
        if np.any(np.isnan(arr)):
            raise ValueError('Data contains NaN values')
        
        if np.any(np.isinf(arr)):
            raise ValueError('Data contains Inf values')
        
        if np.any(arr < min_val) or np.any(arr > max_val):
            raise ValueError(f'Data values must be between {min_val} and {max_val}')
        
        return arr
    
    @staticmethod
    def sanitize(data):
        arr = np.array(data, dtype=np.float32)
        arr = np.clip(arr, -1e6, 1e6)
        arr = np.nan_to_num(arr, nan=0.0, posinf=1e6, neginf=-1e6)
        return arr

# 测试
validator = InputValidator()

# 正常数据
valid_data = [1.0, 2.0, 3.0]
result = validator.validate_numeric_array(valid_data)
print(f'Valid data: {result}')

# 异常数据清洗
bad_data = [1.0, float('nan'), float('inf'), -1e10]
sanitized = validator.sanitize(bad_data)
print(f'Sanitized data: {sanitized}')

## 6. 完整服务示例

In [ ]:
class ResilientInferenceService:
    """带弹性的推理服务"""
    def __init__(self, model_fn):
        self.model_fn = model_fn
        self.circuit_breaker = CircuitBreaker(failure_threshold=3)
        self.rate_limiter = TokenBucketRateLimiter(rate=10, capacity=20)
        self.metrics = MetricsCollector()
        self.validator = InputValidator()
    
    def predict(self, data, client_id='default'):
        # 速率限制
        if not self.rate_limiter.is_allowed():
            self.metrics.inc_counter('requests_total', {'status': 'rate_limited'})
            raise Exception('Rate limit exceeded')
        
        # 熔断检查
        if not self.circuit_breaker.can_execute():
            self.metrics.inc_counter('requests_total', {'status': 'circuit_open'})
            raise Exception('Circuit breaker open')
        
        start = time.time()
        try:
            # 输入验证
            validated = self.validator.validate_numeric_array(data)
            
            # 推理
            result = self.model_fn(validated)
            
            # 记录成功
            self.circuit_breaker.record_success()
            self.metrics.inc_counter('requests_total', {'status': 'success'})
            self.metrics.observe_histogram('latency_seconds', time.time() - start)
            
            return result
        except Exception as e:
            self.circuit_breaker.record_failure()
            self.metrics.inc_counter('requests_total', {'status': 'error'})
            raise

# 测试
def mock_model(x):
    if random.random() < 0.2:
        raise Exception('Model error')
    return x * 2

service = ResilientInferenceService(mock_model)

for i in range(20):
    try:
        result = service.predict([1.0, 2.0, 3.0])
        print(f'Request {i}: Success')
    except Exception as e:
        print(f'Request {i}: {e}')

print('\nMetrics:', service.metrics.get_summary())

## 总结

| 模式 | 用途 | 关键点 |
|:-----|:-----|:-------|
| 熔断器 | 防止级联故障 | 状态机转换 |
| 重试 | 处理临时故障 | 指数退避 |
| 速率限制 | 保护服务 | 令牌桶/滑动窗口 |
| 输入验证 | 安全防护 | 范围检查/清洗 |
| 指标收集 | 可观测性 | 计数器/直方图 |